# Project 7, A team of agents

**AI Fundamentals, Session 3**

Ask a language model for a cover letter and you get a fluent one in seconds. Read it
closely and you sometimes find a skill you do not have, or a language level one step
higher than yours. In a job application, that is not a typo; it is a lie with your name
under it.

In this project several agents work together on one letter. Each is the same local model
with a different job, different instructions and only the input it needs:

```
vacancy ──▶ ANALYST ──▶ requirements ──┐
                                       ├──▶ MATCHER ──▶ evidence ──▶ WRITER ──▶ letter
                     your CV ──────────┘                               ▲           │
                                                                       │           ▼
                                                            feedback ◀── REVIEWER ◀── your CV
```

You will:

- **A** Build one function that turns the model into any agent
- **B** Let the analyst read the vacancy and the matcher compare your CV, and check the matcher with code
- **C** Let the writer draft and the reviewer check every claim, until the reviewer approves
- **D** Compare the team with one big prompt: honesty, time and number of calls
- **E** Use the app with a real vacancy and your own CV

The point: a team of agents is not magic. It is a program that passes messages between
model calls. It helps when a job splits into steps that need different inputs, and when
checking is easier than writing.

In [2]:
import time  # to time every agent
from typing import Literal

import ollama
from pydantic import BaseModel, Field

from settings import CHAT_MODEL, OLLAMA_OPTIONS, check_ollama

check_ollama(CHAT_MODEL)

vacancy = open("data/jobs/vacancy.md", encoding="utf-8").read()
cv = open("data/jobs/cv.md", encoding="utf-8").read()
print(vacancy)

Ollama is running, models ready: qwen3.5:4b
# Junior data analyst, MoveBXL, Brussels

MoveBXL helps the Brussels region plan shared bikes and scooters. We are looking for a
junior data analyst to join our team of six.

## What you will do

- Clean and combine trip data from our partners
- Build dashboards for the mobility department of the region
- Present findings to colleagues who are not data specialists

## What you bring

- A degree in a quantitative field, or equivalent experience
- Python for data analysis, with pandas
- SQL
- Dutch and French at B2 level or higher; English is our working language
- Clear communication with non-specialists

## Nice to have

- Power BI
- Experience with geographic data

Start: January 2027. Hybrid, two days a week at our office near Brussels-North.



In [3]:
print(cv)

# Sam Claes

Brussels, sam.claes@student.example

## Education

- 2026 to 2027: postgraduate programme in data and AI, Erasmushogeschool Brussel (ongoing)
- 2022 to 2025: bachelor in business management. Thesis: when do people in Brussels use
  shared bikes? An analysis of two years of open trip data

## Experience

- 2024 to 2026: customer service agent at a telecom company in Brussels. About 40 calls a
  day, in Dutch and French. Wrote a weekly Excel report on complaint categories for my
  team lead
- Summer 2023: volunteer at a cycling festival in Brussels, planned three routes for 200
  riders on an online map

## Skills

- Python with pandas and matplotlib: course projects and my bachelor thesis
- SQL: basic SELECT queries, in one course project
- Tableau: the dashboards of my thesis
- Excel

## Languages

- Dutch: native
- French: B1, following an evening course to reach B2 by June 2027
- English: C1



Both are fictional, written for this course. Read them as a recruiter would: where does
Sam meet the vacancy, and where not quite? French is B1 where B2 is asked. SQL is basic.
There is Tableau, but no Power BI. A good letter is honest about all three; a bad letter
quietly upgrades them.

---
## Part A, one function, any agent

An agent here is three things: **instructions** (its role), a **task** (the input it
gets) and optionally a **form** for its answer. Everything else is the same model. The
function also keeps a log, so we can count calls and seconds at the end.

In [4]:
log = []   # one entry per call: (agent name, seconds)


def agent(name, instructions, task, form=None):
    """Run the model once in a role. Returns text, or a filled-in form when one is given."""
    start = time.time()
    response = ollama.chat(
        model=CHAT_MODEL,
        messages=[{"role": "system", "content": instructions},
                  {"role": "user", "content": task}],
        format=form.model_json_schema() if form else None,
        options={**OLLAMA_OPTIONS, "temperature": 0, "num_predict": 1500},
        think=False,
    )
    seconds = time.time() - start
    log.append((name, seconds))
    print(f"  {name:<9} {seconds:5.1f} s")
    return form.model_validate_json(response.message.content) if form else response.message.content

---
## Part B, the analyst and the matcher

The **analyst** sees only the vacancy and turns it into a list of requirements. It does
not see the CV, so it cannot bend the requirements towards it.

In [5]:
class Requirement(BaseModel):
    requirement: str
    must_have: bool = Field(description="false for nice-to-have")


class Requirements(BaseModel):
    company: str
    role: str
    requirements: list[Requirement]


ANALYST = """You analyse job vacancies. List every requirement separately, in a few words each:
the must-haves and the nice-to-haves."""

needs = agent("analyst", ANALYST, f"Vacancy:\n\n{vacancy}", Requirements)
print(f"\n{needs.role} at {needs.company}")
for r in needs.requirements:
    print(f"  {'must' if r.must_have else 'nice'}  {r.requirement}")

  analyst     6.9 s

Junior Data Analyst at MoveBXL
  must  Degree quantitative field
  must  Python pandas
  must  SQL
  must  Dutch B2+
  must  French B2+
  must  English working language
  must  Communication non-specialists
  nice  Power BI
  nice  Geographic data experience


The **matcher** gets the requirements and the CV, and nothing else. For every requirement
it must quote the CV, and only then give its verdict. Evidence before verdict, for the
same reason as in Project 4: the model writes left to right, so whatever comes first
shapes what follows.

In [6]:
class Match(BaseModel):
    requirement: str
    evidence: str = Field(description="the exact words from the CV that show it, or an empty string")
    verdict: Literal["met", "partly", "missing"]


class Matches(BaseModel):
    candidate: str = Field(description="the candidate's full name, from the CV")
    matches: list[Match]


MATCHER = """You compare a CV with job requirements. Be strict: a level below the one asked
is partly, not met. Only quote words that really are in the CV."""

requirement_list = "\n".join(f"- {r.requirement} ({'must have' if r.must_have else 'nice to have'})"
                             for r in needs.requirements)
found = agent("matcher", MATCHER, f"Requirements:\n{requirement_list}\n\nCV:\n\n{cv}", Matches)

for m in found.matches:
    print(f"  {m.verdict:<8} {m.requirement[:40]:<42} {m.evidence[:60]!r}")

  matcher    13.2 s
  met      Degree quantitative field (must have)      '2026 to 2027: postgraduate programme in data and AI, Erasmus'
  met      Python pandas (must have)                  'Python with pandas and matplotlib'
  partly   SQL (must have)                            'SQL: basic SELECT queries, in one course project'
  met      Dutch B2+ (must have)                      'Dutch: native'
  partly   French B2+ (must have)                     'French: B1, following an evening course to reach B2 by June '
  met      English working language (must have)       'English: C1'
  met      Communication non-specialists (must have   'customer service agent at a telecom company in Brussels. Abo'
  partly   Power BI (nice to have)                    ''
  met      Geographic data experience (nice to have   'Summer 2023: volunteer at a cycling festival in Brussels, pl'


### Check the matcher with code

The matcher claims to quote the CV. That is a claim code can check: is the quote really
in the CV? A verdict without a real quote becomes *missing*. This one check removes a
whole class of invention before it can reach the letter.

In [7]:
def normalise(text):
    """Lower case, and every run of spaces and line breaks becomes one space."""
    return " ".join(text.lower().split())


for m in found.matches:
    quoted = m.evidence.strip() != "" and normalise(m.evidence) in normalise(cv)
    if m.verdict != "missing" and not quoted:
        print(f"  corrected: {m.requirement!r} was {m.verdict!r} without a real quote, now 'missing'")
        m.verdict = "missing"

evidence = "\n".join(f"- {m.requirement}: {m.verdict}. The CV says: {m.evidence}"
                     for m in found.matches if m.verdict != "missing")
print()
print(evidence)

  corrected: 'Power BI (nice to have)' was 'partly' without a real quote, now 'missing'

- Degree quantitative field (must have): met. The CV says: 2026 to 2027: postgraduate programme in data and AI, Erasmushogeschool Brussel (ongoing)
- Python pandas (must have): met. The CV says: Python with pandas and matplotlib
- SQL (must have): partly. The CV says: SQL: basic SELECT queries, in one course project
- Dutch B2+ (must have): met. The CV says: Dutch: native
- French B2+ (must have): partly. The CV says: French: B1, following an evening course to reach B2 by June 2027
- English working language (must have): met. The CV says: English: C1
- Communication non-specialists (must have): met. The CV says: customer service agent at a telecom company in Brussels. About 40 calls a day, in Dutch and French.
- Geographic data experience (nice to have): met. The CV says: Summer 2023: volunteer at a cycling festival in Brussels, planned three routes for 200 riders on an online map


---
## Part C, the writer and the reviewer

The **writer** never sees the CV. It sees the evidence list, which contains only what the
matcher could quote. The **reviewer** sees the letter and the CV, and checks every claim
the letter makes about Sam. If anything is unsupported, the writer gets the feedback and
tries again, at most three times.

Why a separate reviewer, if it is the same model? Because **checking is easier than
writing**. The writer has to produce 170 fluent words and keep every fact straight at the
same time. The reviewer has one narrow question per sentence: *does the CV say this?*

In [8]:
WRITER = """You write cover letters. Use only the facts in the evidence list. Never claim a
skill, a level or an experience that the evidence does not show. Where a requirement is
only partly met, say so honestly, or leave it out. At most 170 words, no addresses at the top."""

REVIEWER = """You check a cover letter against the CV it is based on. List every claim the
letter makes about the candidate's education, experience, skills and languages, and check
each one against the CV. A claim is supported only if the CV says it.
Ignore polite phrases, motivation, and facts about the company or the job."""


class Claim(BaseModel):
    claim: str = Field(description="one thing the letter says about the candidate")
    cv_evidence: str = Field(description="the exact words from the CV that support it, or an empty string")
    supported: bool


class Review(BaseModel):
    claims: list[Claim]
    feedback: str = Field(description="what the writer must change, or an empty string")
    approved: bool = Field(description="true only if every claim is supported")


def write_and_review(max_rounds=3):
    """Draft, review, revise. Returns the last letter and the last review."""
    brief = f"Role: {needs.role} at {needs.company}.\nCandidate: {found.candidate}.\n\nEvidence:\n{evidence}"
    letter = agent("writer", WRITER, brief)
    for round_number in range(1, max_rounds + 1):
        review = agent("reviewer", REVIEWER, f"CV:\n\n{cv}\n\nLetter:\n\n{letter}", Review)
        unsupported = [c.claim for c in review.claims if not c.supported]
        print(f"  round {round_number}: {len(review.claims)} claims checked, {len(unsupported)} unsupported")
        for claim in unsupported:
            print(f"     ✗ {claim}")
        if review.approved and not unsupported:
            return letter, review
        letter = agent("writer", WRITER, f"{brief}\n\nYour previous letter:\n{letter}\n\n"
                                         f"The reviewer says:\n{review.feedback}\n\nWrite the corrected letter.")
    return letter, review


letter, review = write_and_review()
print("\n" + letter)

  writer      6.4 s


  reviewer   15.4 s
  round 1: 9 claims checked, 0 unsupported

Dear Hiring Manager,

I am writing to apply for the Junior Data Analyst position at MoveBXL. My background aligns closely with your core requirements. I hold a C1 level in English and am a native Dutch speaker, ensuring effective communication with non-specialists. I have extensive experience as a customer service agent handling 40 daily calls in both languages.

I possess the required Python skills using pandas and matplotlib. While my SQL proficiency is currently basic, limited to SELECT queries from one course project, I am actively developing these skills through my ongoing postgraduate programme in data and AI at Erasmus Hogeschool Brussel (2026–2027).

Regarding French, I hold a B1 level and am following an evening course to reach B2 by June 2027. Additionally, I have practical experience with geographic data as a volunteer who planned three cycling routes for 200 riders using an online map in summer 2023.

I am eage

Read the rounds above. If the reviewer rejected a draft, read what it caught: usually a
skill nudged one step up, or a tool that is not in the CV. Then read the final letter
yourself. The reviewer is a model too, and the last word belongs to you.

**Exercise C1.** Change Sam's CV so that French is B2. Run Parts B and C again. Does the
letter change, and should it?

**Exercise C2.** Make the writer greedier: add *Make the candidate sound as strong as
possible* to `WRITER`. How many rounds does the reviewer need now?

---
## Part D, the team against one big prompt

The obvious alternative: give the model the vacancy and the CV together and ask for the
letter in one go. Then let the same reviewer judge that letter.

In [9]:
team_calls, team_seconds = len(log), sum(s for _, s in log)

single = agent("single", "You are a helpful assistant.",
               f"Write a cover letter of at most 170 words for this vacancy, based on my CV."
               f"\n\nVacancy:\n{vacancy}\n\nCV:\n{cv}")
print("\n" + single + "\n")

single_review = agent("reviewer", REVIEWER, f"CV:\n\n{cv}\n\nLetter:\n\n{single}", Review)
for c in single_review.claims:
    print(f"  {'✓' if c.supported else '✗'} {c.claim}")

  single      6.8 s

Dear Hiring Team at MoveBXL,

I am writing to express my strong interest in the Junior Data Analyst position. As a current postgraduate student in Data and AI at Erasmushogeschool Brussel, I possess the quantitative foundation required for this role, complemented by direct experience analyzing shared bike data through my bachelor's thesis on Brussels mobility patterns.

My technical skills align perfectly with your needs. I am proficient in Python (pandas) and have executed basic SQL queries, alongside creating interactive dashboards using Tableau. While my French is currently B1, I am actively following evening courses to reach B2 by June 2027, ensuring full fluency for our team. My native Dutch and C1 English allow me to communicate complex findings clearly to non-specialists, a skill honed during my role as a customer service agent where I produced weekly reports on complaint categories.

Beyond data, my volunteer experience planning cycling routes demonstrates 

  reviewer   23.0 s
  ✓ I am a current postgraduate student in Data and AI at Erasmushogeschool Brussel
  ✗ I possess the quantitative foundation required for this role
  ✓ complemented by direct experience analyzing shared bike data through my bachelor's thesis on Brussels mobility patterns
  ✓ I am proficient in Python (pandas)
  ✓ I have executed basic SQL queries
  ✓ creating interactive dashboards using Tableau
  ✓ My French is currently B1
  ✓ I am actively following evening courses to reach B2 by June 2027
  ✓ My native Dutch and C1 English allow me to communicate complex findings clearly to non-specialists
  ✓ a skill honed during my role as a customer service agent where I produced weekly reports on complaint categories
  ✓ my volunteer experience planning cycling routes demonstrates my understanding of the mobility sector's context
  ✗ I am eager to bring this blend of technical ability and domain awareness to MoveBXL, starting January 2027


In [10]:
unsupported_team = sum(not c.supported for c in review.claims)
unsupported_single = sum(not c.supported for c in single_review.claims)
print(f"{'':<14}{'calls':>6}{'seconds':>9}{'unsupported claims':>21}")
print(f"{'team':<14}{team_calls:>6}{team_seconds:>9.0f}{unsupported_team:>21}")
print(f"{'one prompt':<14}{1:>6}{log[-2][1]:>9.0f}{unsupported_single:>21}")

               calls  seconds   unsupported claims
team               4       42                    0
one prompt         1        7                    2


The team costs several times more calls and time. Whether it buys anything shows in the
last column, and it varies from run to run and model to model: sometimes the single prompt
is honest too. That variation is the real argument for the reviewer. You cannot know in
advance which run invents something, so you check every run.

When is a team of agents worth it?

| A team helps when | One prompt is enough when |
| --- | --- |
| steps need different inputs, like the writer who must not see the CV | the task is short and fits in one instruction |
| you want to inspect each step, like the evidence list | you will read and correct the result yourself |
| a separate check catches errors, because checking is easier than writing | the cost of a mistake is low |
| parts can run with different tools or models | speed and cost matter most |

And the cost side: more calls, more waiting, more code, and an early mistake travels down
the line. The matcher's error becomes the writer's error unless something stops it, which
is why we checked its quotes with code.

---
## Part E, the app: your own vacancy, your own CV

The app runs the whole team and shows each agent's work as it happens. It starts with Sam;
replace both texts with a real vacancy and your own CV. Everything stays on your laptop.

In [11]:
import gradio as gr


def run_team(vacancy_text, cv_text):
    """Run the whole team. 'yield' sends progress to the page after every step."""
    global vacancy, cv, needs, found, evidence     # the functions above read these names
    vacancy, cv = vacancy_text, cv_text
    report = ""

    needs = agent("analyst", ANALYST, f"Vacancy:\n\n{vacancy}", Requirements)
    report += f"### Analyst\n{len(needs.requirements)} requirements for **{needs.role}** at {needs.company}\n\n"
    yield report, ""

    found = agent("matcher", MATCHER, "Requirements:\n" + "\n".join(
        f"- {r.requirement} ({'must have' if r.must_have else 'nice to have'})" for r in needs.requirements)
        + f"\n\nCV:\n\n{cv}", Matches)
    for m in found.matches:                       # the same code check as in Part B
        if m.verdict != "missing" and not (m.evidence.strip() and normalise(m.evidence) in normalise(cv)):
            m.verdict = "missing"
    icons = {"met": "✅", "partly": "🟡", "missing": "❌"}
    report += "### Matcher\n" + "\n".join(f"- {icons[m.verdict]} {m.requirement}" for m in found.matches) + "\n\n"
    yield report, ""

    evidence = "\n".join(f"- {m.requirement}: {m.verdict}. The CV says: {m.evidence}"
                         for m in found.matches if m.verdict != "missing")
    letter, last_review = write_and_review()
    problems = [c.claim for c in last_review.claims if not c.supported]
    report += "### Writer and reviewer\n" + ("Approved: every claim is in the CV." if not problems else
                                             "Still unsupported after three rounds:\n" + "\n".join(f"- {p}" for p in problems))
    yield report, letter


with gr.Blocks(title="Cover letter team") as team_app:
    gr.Markdown("## A cover letter that does not lie")
    with gr.Row():
        vacancy_box = gr.Textbox(value=vacancy, label="Vacancy", lines=14)
        cv_box = gr.Textbox(value=cv, label="Your CV", lines=14)
    button = gr.Button("Write my letter", variant="primary")
    with gr.Row():
        progress = gr.Markdown()
        letter_box = gr.Textbox(label="Letter", lines=14)
    button.click(run_team, inputs=[vacancy_box, cv_box], outputs=[progress, letter_box])

team_app.launch(inbrowser=True, inline=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [12]:
team_app.close()

Closing server running on port: 7860


**Exercise E1.** Add a fifth agent: a **translator** that turns the approved letter into
Dutch or French. Should the reviewer check the translation too?

**Exercise E2.** Which agent would you give a tool from Project 6, and which tool? A
`read_note` for the matcher, so it can use your notes as well as your CV?

---
## What to take from this

**A team of agents is ordinary code.** Four model calls with different instructions,
passing structured results to each other. Frameworks for agents add conveniences, but
underneath they do what you just wrote.

**Give each agent only what it needs.** The analyst never saw the CV and the writer never
saw it either. What an agent cannot see, it cannot bend or invent from.

**Checking is easier than writing.** A reviewer with one narrow question catches what the
writer let slip. Code catches even more: a quote is either in the CV or it is not.

**It costs.** Several times the calls and the time of one prompt. Use a team when the
steps, the inputs or the checks are genuinely different, not because it sounds advanced.